# Выбор старта RL: адаптер SFT и температура (EXP-019)

Проверка генерацией (EXP-015, `sft_eval.ipynb`) показала: адаптер эпохи 3 из 7 не проходит правило приёмки SFT
(награда ниже исходной T-lite в обоих режимах, больше `#SOURCE` не из выдачи), петли — в 13 ответах из 90;
top-k 0 не прошёл EXP-017. Здесь — сетка на тех же 25 темах validation, по 4 ответа на тему и режим (200 эпизодов
на конфигурацию): исходная T-lite (`base`), адаптер эпохи 3 из 7 (`sft7`, `runs/sft/20261107_1913`) и эпохи 3 из 3
(`sft3`, `runs/sft/20261107_1525`) × T 0.8 и 0.9, без top-k и top-p, как в RL. Сиды — по теме, режиму и номеру
ответа, одинаковые у всех конфигураций. Предел ответа 8000 токенов, контекста — 25 000 (как в RL).

Правило (`docs/decisions.md`, раздел 9, решение 11.11.2026 — до прогона). Конфигурация допускается, если против `base`
при той же температуре: не хуже по доле корректных строк (оба режима) и по `#SOURCE` не из выдачи; лучше хотя бы
по одному из — заключение последним слайдом, число слайдов в диапазоне, вызов поиска при «Поиск: есть»; не хуже
по награде (оба режима); разброс награды внутри темы сузился не больше чем вдвое; доля вырожденных ответов — не больше,
чем у `base`, плюс 5 п.п. (10 из 200) «Хуже» и «лучше» — интервал бутстрепа по темам целиком по одну сторону от нуля (разница
средних по 4 ответам темы). Среди допущенных — меньшая доля вырожденных; если у нескольких она в пределах 2 п.п.
от лучшей — из них большая средняя награда. Что делать, если не допущена ни одна, решаю после прогона.

Генерация — два сервера vLLM (по одному на карту, у каждого исходная модель и оба адаптера); оценка — после
остановки серверов. Результат — `runs/sft_select/<RUN>/<конфигурация>/`, `episodes.csv`, `comparison.csv`,
`selection.csv`.

## 1. Серверы vLLM (терминал)

Карта 0 — T 0.8 (порт 8000), карта 1 — T 0.9 (порт 8001); температура задаётся в запросе, разделение по картам —
только чтобы шли параллельно. Окружение `~/vllm085` (раздел 2 `grpo.ipynb`).
```bash
source ~/vllm085/bin/activate
cd ~/Diplom
for GPU in 0 1; do
VLLM_USE_V1=0 CUDA_VISIBLE_DEVICES=$GPU nohup python -m vllm.entrypoints.openai.api_server \
    --model t-tech/T-lite-it-2.1 --revision main --served-model-name base --dtype float16 \
    --enable-lora --max-lora-rank 32 --max-loras 2 \
    --lora-modules sft7=/home/jovyan/Diplom/runs/sft/20261107_1913/adapter sft3=/home/jovyan/Diplom/runs/sft/20261107_1525/adapter \
    --max-model-len 25600 --gpu-memory-utilization 0.85 --generation-config vllm --port 800$GPU --seed 0 \
    > ~/vllm_select_$GPU.log 2>&1 &
done
grep -h "startup complete" ~/vllm_select_*.log   # готовы оба — две строки; ошибки — tail ~/vllm_select_0.log
```
Остановить оба: `pkill -f vllm.entrypoints.openai.api_server`.

In [ ]:
import json, os, statistics, sys, threading, time, traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from transformers import AutoTokenizer

from presentation_rl.environment import run_episode
from presentation_rl.evaluation import degeneration, evaluate_folder, paired, reward_spread, topic_means, write_csv
from presentation_rl.rewards import load_key_points
from presentation_rl.tasks import load_tasks
from presentation_rl.tools.cache import WikiCache
from presentation_rl.tools.wiki import WikiClient
from presentation_rl.tools.wiki_tools import WikiTools
from presentation_rl.training.common import LockedTokenizer, episode_prompt, mode_task, stop_ids, vllm_generator
from presentation_rl.training.vllm_client import VLLMClient

os.environ["WIKI_PROXY"] = ""  # Википедия открыта в пространстве; туннель — docs/server.md
RUN = "20261111"
OUT = ROOT / "runs" / "sft_select" / RUN
MODEL_ID = "t-tech/T-lite-it-2.1"
MODELS = ("base", "sft7", "sft3")  # имена на серверах vLLM: исходная модель и адаптеры --lora-modules
TEMPERATURES = (0.8, 0.9)
SERVERS = {0.8: "http://127.0.0.1:8000", 0.9: "http://127.0.0.1:8001"}
SAMPLES, MAX_NEW_TOKENS, MAX_CONTEXT, WORKERS = 4, 8000, 25000, 48
MODES = ("search", "plain")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision="main")
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="validation")}
points = load_key_points(ROOT / "data" / "tasks" / "key_points" / "key_points.json")
IDS = sorted(tasks)
EPISODES = [(mode, key, s) for mode in MODES for key in IDS for s in range(SAMPLES)]
CONFIGS = [(f"{m}_T{round(t * 10):02d}", m, t) for t in TEMPERATURES for m in MODELS]
cache = WikiCache(ROOT / "data" / "sources" / "wiki_cache",
                  WikiClient(contact=os.environ.get("WIKI_CONTACT", "https://github.com/doshik722")))
print(f"тем {len(IDS)}, эпизодов на конфигурацию {len(EPISODES)}, конфигураций {len(CONFIGS)}")
for t, url in SERVERS.items():
    print(f"T {t}: {url}", VLLMClient(url).models())

## 2. Генерация

Каждый сервер — свои три конфигурации, эпизоды всех трёх идут вперемешку (`WORKERS` одновременно; пакеты собирает
vLLM). Эпизод сохраняется сразу: `.json` (конец, токены модели, вызовы, ошибка, время), `.txt` (эпизод после промпта),
`_answer.txt`. При перезапуске готовые эпизоды пропускаются, эпизоды с ошибкой повторяются.

In [ ]:
safe = LockedTokenizer(tokenizer)
prompt = episode_prompt(tokenizer)
PROMPTS = {(mode, key): prompt(mode_task(tasks[key], mode).request) for mode in MODES for key in IDS}  # основной поток
STOPS = stop_ids(tokenizer)


def done(folder, stem):
    path = folder / f"{stem}.json"
    return path.exists() and not json.loads(path.read_text(encoding="utf-8")).get("error")


def one_episode(client, name, model, temperature, mode, key, sample):
    folder, stem = OUT / name, f"{mode}_{key}_{sample}"
    row, start = {"mode": mode, "id": key, "sample": sample}, time.monotonic()
    try:
        prompt_ids = PROMPTS[(mode, key)]
        generate = vllm_generator(client, model, temperature, STOPS, ("sft_select", mode, key, sample))
        episode = run_episode(prompt_ids, safe, generate, WikiTools(cache) if mode == "search" else None,
                              max_new_tokens=MAX_NEW_TOKENS, max_context=MAX_CONTEXT)
        (folder / f"{stem}.txt").write_text(safe.decode(episode.ids[len(prompt_ids):], skip_special_tokens=False),
                                            encoding="utf-8")
        (folder / f"{stem}_answer.txt").write_text(episode.answer, encoding="utf-8")
        row.update(finish=episode.finish, generated_tokens=episode.generated_tokens, calls=len(episode.calls),
                   truncated_outputs=episode.truncated_outputs, error="")
    except Exception:
        row.update(error=traceback.format_exc()[-1500:])
    row["seconds"] = round(time.monotonic() - start)
    (folder / f"{stem}.json").write_text(json.dumps(row, ensure_ascii=False, indent=1), encoding="utf-8")
    return name, row


def run_server(temperature):
    client = VLLMClient(SERVERS[temperature])
    jobs = []
    for name, model, t in CONFIGS:
        if t == temperature:
            (OUT / name).mkdir(parents=True, exist_ok=True)
            jobs += [(name, model, mode, key, s) for mode, key, s in EPISODES if not done(OUT / name, f"{mode}_{key}_{s}")]
    jobs.sort(key=lambda j: (j[4], j[3], j[2], j[0]))  # конфигурации вперемешку: сравнимый ход у всех трёх
    counts, errors, start = {}, 0, time.monotonic()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = [pool.submit(one_episode, client, name, model, temperature, mode, key, s)
                   for name, model, mode, key, s in jobs]
        for k, future in enumerate(futures, 1):
            name, row = future.result()
            counts[name] = counts.get(name, 0) + 1
            errors += bool(row["error"])
            if k % 50 == 0 or k == len(futures):
                print(f"T {temperature}: {k}/{len(futures)} за {(time.monotonic() - start) / 60:.0f} мин, ошибок {errors}",
                      flush=True)


threads = [threading.Thread(target=run_server, args=(t,)) for t in TEMPERATURES]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

## 3. Оценка

После генерации: остановить серверы (`pkill -f vllm.entrypoints.openai.api_server`) — bge-m3 занимает карту 1.

In [ ]:
from presentation_rl.rewards import Lemmatizer, SentenceEmbedder

embedder, lemmatizer = SentenceEmbedder("BAAI/bge-m3", device="cuda:1"), Lemmatizer()
rows = []
for name, model, t in CONFIGS:
    part = evaluate_folder(OUT / name, name, tasks, points, embedder, lemmatizer, build_pptx=False)
    print(f"{name}: эпизодов {len(part)}, ошибок {sum(bool(r.get('error')) for r in part)}")
    rows += part
write_csv(OUT / "episodes.csv", rows)

## 4. Допуск и выбор

По каждой конфигурации — против `base` той же температуры; отдельно — влияние температуры на одной модели
(T 0.8 против 0.9). Сверка движков: `sft7_T09` здесь (vLLM) и SFT в `sft_eval` (`transformers`, первый ответ на тему).

In [ ]:
METRICS = ["parse_rate", "slides", "in_range", "conclusion", "stop", "searched", "calls", "sources_ok", "sources_bad",
           "images", "image_words", "images_related", "generated_tokens", "degenerate", "loop_8gram", "reward_total",
           "reward_coverage", "reward_keys", "reward_length", "reward_visual", "reward_format"]


def share(name, key="degenerate", mode=None):
    part = [r for r in rows if r["model"] == name and not r.get("error") and (mode is None or r["mode"] == mode)]
    return statistics.mean(bool(r[key]) for r in part) if part else None


def mean_reward(name):  # средняя по темам награда, среднее двух режимов
    means = [topic_means(rows, name, "reward_total", mode) for mode in MODES]
    return statistics.mean(statistics.mean(m.values()) for m in means if m) if any(means) else None


def admission(name, base):
    test = lambda metric, mode: paired(rows, metric, mode, base, name)
    worse = lambda t: t is not None and t["high"] < 0
    better = lambda t: t is not None and t["low"] > 0
    bad_sources = test("sources_bad", "search")
    spread_base, spread = reward_spread(rows, base), reward_spread(rows, name)
    checks = {
        "ok_parse_rate": not any(worse(test("parse_rate", m)) for m in MODES),
        "ok_sources_bad": not (bad_sources is not None and bad_sources["low"] > 0),
        "ok_better_one": any(better(test(metric, mode)) for metric, mode in
                          (("conclusion", "plain"), ("conclusion", "search"), ("in_range", "plain"),
                           ("in_range", "search"), ("searched", "search"))),
        "ok_reward": not any(worse(test("reward_total", m)) for m in MODES),
        "ok_spread": spread is not None and spread_base is not None and spread >= spread_base / 2,
        "ok_degeneration": share(name) <= share(base) + 0.05,
    }
    return checks


table, selection = [], []
for name, model, t in CONFIGS:
    base = f"base_T{round(t * 10):02d}"
    row = {"config": name, "degenerate": share(name), "degenerate_search": share(name, mode="search"),
           "degenerate_plain": share(name, mode="plain"), "unfinished": share(name, "unfinished"),
           "reward": mean_reward(name), "spread": reward_spread(rows, name),
           "tokens_median": statistics.median(r["generated_tokens"] for r in rows if r["model"] == name and not r.get("error"))}
    if name != base:
        checks = admission(name, base)
        row.update(checks, admitted=all(checks.values()))
        table += [{"config": name, "against": base, **c} for mode in MODES for metric in METRICS
                  if (c := paired(rows, metric, mode, base, name))]
    selection.append(row)
for model in MODELS:  # влияние температуры на одной модели: T 0.8 − T 0.9
    table += [{"config": f"{model}_T08", "against": f"{model}_T09", **c} for mode in MODES for metric in METRICS
              if (c := paired(rows, metric, mode, f"{model}_T09", f"{model}_T08"))]
write_csv(OUT / "comparison.csv", table)
write_csv(OUT / "selection.csv", selection)
for r in selection:
    print("  ".join(f"{k} {v:.3f}" if isinstance(v, float) else f"{k} {v}" for k, v in r.items()))

admitted = [r for r in selection if r.get("admitted")]
if admitted:
    best = min(r["degenerate"] for r in admitted)
    close = [r for r in admitted if r["degenerate"] <= best + 0.02]
    print("Выбор по правилу:", max(close, key=lambda r: r["reward"])["config"], "| допущены:", [r["config"] for r in admitted])
else:
    print("Не допущена ни одна конфигурация — решение после разбора (правило, п. «если не допущена ни одна»)")

In [ ]:
# Сверка движков: sft7 при T 0.9 через vLLM против SFT в sft_eval через transformers (первый ответ на тему)
hf = ROOT / "runs" / "sft_eval" / "20261107" / "sft"
hf_rows = []
for mode, key, s in [(m, k, 0) for m in MODES for k in IDS]:
    meta = hf / f"{mode}_{key}_{s}.json"
    if meta.exists():
        info = json.loads(meta.read_text(encoding="utf-8"))
        if not info.get("error"):
            answer = (hf / f"{mode}_{key}_{s}_answer.txt").read_text(encoding="utf-8")
            hf_rows.append({**info, **degeneration(answer, info["finish"])})
vllm_rows = [r for r in rows if r["model"] == "sft7_T09" and r["sample"] == 0 and not r.get("error")]
for label, part in (("transformers", hf_rows), ("vLLM", vllm_rows)):
    print(f"{label:12s} ответов {len(part)}, вырожденных {statistics.mean(bool(r['degenerate']) for r in part):.2f}, "
          f"не закончено {statistics.mean(r['finish'] != 'stop' for r in part):.2f}, "
          f"токенов медиана {statistics.median(r['generated_tokens'] for r in part):.0f}")